In [ ]:
import sys
from pathlib import Path
import sqlite3
import random
import io

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image

ROOT = Path("__file__").resolve().parents[3]
CODE_DIR = Path("__file__").resolve().parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE_DIR))

from ccg_card_id.config import cfg
from db import open_db

In [ ]:
DATA_DIR = cfg.data_dir
DB_PATH  = DATA_DIR / "datasets" / "packopening" / "packopening.db"

con = open_db(DB_PATH)
print(f"DB: {DB_PATH}")
print(f"Data dir: {DATA_DIR}")

In [ ]:
# Load all frames that have a phash_dist computed
rows = con.execute(
    "SELECT id, frame_path, aligned_path, card_id, set_code, phash_dist, human_review "
    "FROM frames WHERE phash_dist IS NOT NULL ORDER BY phash_dist"
).fetchall()

ids      = np.array([r["id"]          for r in rows], dtype=np.int32)
paths    = [r["frame_path"]            for r in rows]
aligned  = [r["aligned_path"]          for r in rows]
card_ids = [r["card_id"]               for r in rows]
set_codes= [r["set_code"] or ""        for r in rows]
dists    = np.array([r["phash_dist"]   for r in rows], dtype=np.int32)
reviews  = [r["human_review"]          for r in rows]

print(f"Frames with phash_dist: {len(rows)}")
print(f"  dist range : {dists.min()} – {dists.max()}")
print(f"  median dist: {np.median(dists):.1f}")
human_reviewed = sum(1 for r in reviews if r is not None)
print(f"  human-reviewed: {human_reviewed} / {len(rows)}")

In [ ]:
THUMB_SIZE = (224, 312)   # w × h thumbnail shown in notebook
N_SAMPLES  = 6            # number of sample images per band


def _load_thumb(rel_path: str | None, fallback_rel: str | None = None) -> bytes | None:
    """Return JPEG bytes for a thumbnail, trying aligned_path then frame_path."""
    for p in [rel_path, fallback_rel]:
        if not p:
            continue
        full = DATA_DIR / p
        if full.exists():
            try:
                img = Image.open(full).convert("RGB")
                img.thumbnail(THUMB_SIZE, Image.LANCZOS)
                buf = io.BytesIO()
                img.save(buf, format="JPEG", quality=80)
                return buf.getvalue()
            except Exception:
                pass
    return None


def mark_review(frame_id: int, verdict: str | None) -> None:
    """Persist human_review verdict ('good', 'bad', or NULL) to DB."""
    con.execute(
        "UPDATE frames SET human_review=? WHERE id=?",
        (verdict, frame_id),
    )
    con.commit()
    # Update local cache too
    idx = list(ids).index(frame_id)
    reviews[idx] = verdict


def make_card_widget(idx: int) -> widgets.VBox:
    """Return a VBox with thumbnail, info label, and Good/Bad/Clear buttons."""
    frame_id  = int(ids[idx])
    dist      = int(dists[idx])
    current   = reviews[idx]
    set_code  = set_codes[idx]
    card_id   = card_ids[idx]

    img_bytes = _load_thumb(aligned[idx], paths[idx])
    if img_bytes:
        img_widget = widgets.Image(value=img_bytes, format="jpeg",
                                   layout=widgets.Layout(width="160px"))
    else:
        img_widget = widgets.Label("[no image]")

    review_label = widgets.Label(
        f"dist={dist}  {set_code}  review={current or '—'}",
        layout=widgets.Layout(font_size="11px")
    )

    btn_good  = widgets.Button(description="Good",  button_style="success",
                               layout=widgets.Layout(width="60px", height="28px"))
    btn_bad   = widgets.Button(description="Bad",   button_style="danger",
                               layout=widgets.Layout(width="60px", height="28px"))
    btn_clear = widgets.Button(description="Clear", button_style="",
                               layout=widgets.Layout(width="60px", height="28px"))

    def _on_good(_):  mark_review(frame_id, "good");  review_label.value = f"dist={dist}  {set_code}  review=good"
    def _on_bad(_):   mark_review(frame_id, "bad");   review_label.value = f"dist={dist}  {set_code}  review=bad"
    def _on_clear(_): mark_review(frame_id, None);    review_label.value = f"dist={dist}  {set_code}  review=—"

    btn_good.on_click(_on_good)
    btn_bad.on_click(_on_bad)
    btn_clear.on_click(_on_clear)

    btns = widgets.HBox([btn_good, btn_bad, btn_clear])
    return widgets.VBox(
        [img_widget, review_label, btns],
        layout=widgets.Layout(border="1px solid #ccc", padding="6px", margin="4px")
    )

In [ ]:
# ── Interactive review widget ────────────────────────────────────────────────

slider = widgets.IntSlider(
    value=10, min=0, max=64, step=1,
    description="Threshold:",
    continuous_update=False,
    layout=widgets.Layout(width="500px"),
)

out_hist    = widgets.Output()
out_keep    = widgets.Output()
out_filter  = widgets.Output()
out_stats   = widgets.Output()


def _sample_near(threshold: int, above: bool, n: int = N_SAMPLES) -> list[int]:
    """Return up to n indices with dists just below (above=False) or just above (above=True) threshold."""
    if above:
        cands = np.where(dists >= threshold)[0]
        if len(cands) == 0:
            return []
        # Sort by dist ascending to get closest-to-threshold first
        order = np.argsort(dists[cands])
        cands = cands[order]
    else:
        cands = np.where(dists < threshold)[0]
        if len(cands) == 0:
            return []
        # Sort by dist descending to get closest-to-threshold first
        order = np.argsort(-dists[cands])
        cands = cands[order]
    # Take up to n, but spread them — pick first n unique set_code reps if possible
    seen_sets: set[str] = set()
    chosen = []
    for i in cands:
        if set_codes[i] not in seen_sets:
            chosen.append(int(i))
            seen_sets.add(set_codes[i])
        if len(chosen) >= n:
            break
    # Fill remaining slots if not enough diverse sets
    for i in cands:
        if len(chosen) >= n:
            break
        if int(i) not in chosen:
            chosen.append(int(i))
    return chosen


def refresh(threshold: int) -> None:
    n_keep   = int((dists < threshold).sum())
    n_filter = int((dists >= threshold).sum())

    # ── Histogram ──────────────────────────────────────────────────────────
    with out_hist:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 3))
        bins = np.arange(0, 65, 1)
        ax.hist(dists, bins=bins, color="steelblue", edgecolor="none", alpha=0.8)
        ax.axvline(threshold, color="crimson", lw=2, label=f"threshold={threshold}")
        ax.set_xlabel("pHash Hamming distance")
        ax.set_ylabel("frames")
        ax.set_title("pHash distance distribution")
        ax.legend()
        plt.tight_layout()
        plt.show()

    # ── Stats ──────────────────────────────────────────────────────────────
    with out_stats:
        clear_output(wait=True)
        hr_good = sum(1 for r in reviews if r == "good")
        hr_bad  = sum(1 for r in reviews if r == "bad")
        print(
            f"Threshold {threshold}:  keep {n_keep} ({100*n_keep/max(1,len(dists)):.1f}%)  "
            f"filter {n_filter} ({100*n_filter/max(1,len(dists)):.1f}%)   "
            f"| human-reviewed: {hr_good} good / {hr_bad} bad"
        )

    # ── Samples just below threshold ("would keep") ────────────────────────
    with out_keep:
        clear_output(wait=True)
        print(f"─── Would KEEP (dist < {threshold}) — closest to threshold ───")
        idxs = _sample_near(threshold, above=False)
        if not idxs:
            print("  (none)")
        else:
            display(widgets.HBox([make_card_widget(i) for i in idxs]))

    # ── Samples just above threshold ("would filter") ──────────────────────
    with out_filter:
        clear_output(wait=True)
        print(f"─── Would FILTER (dist ≥ {threshold}) — closest to threshold ───")
        idxs = _sample_near(threshold, above=True)
        if not idxs:
            print("  (none)")
        else:
            display(widgets.HBox([make_card_widget(i) for i in idxs]))


def _on_slider(change):
    refresh(change["new"])


slider.observe(_on_slider, names="value")

display(widgets.VBox([
    slider,
    out_stats,
    out_hist,
    out_keep,
    out_filter,
]))

refresh(slider.value)